# Ensemble Techniques: Bagging and Boosting

## 📚 Learning Objectives

By completing this notebook, you will:
- Understand ensemble methods (Bagging, Boosting)
- Experiment with ensemble techniques to improve model performance
- Compare bagging vs boosting approaches
- Apply ensemble methods to classification problems

## 🔗 Where this fits

**Builds on:** Course 04 — Unit 3, lesson 02 "Decision Trees and Random Forest" — one tree overfits; bagging and boosting are two ways of combining many.

**Used later in:** Course 04 — Unit 5, lesson 02, which tunes gradient boosting properly.

---

This notebook covers practical activities from **Course 04, Unit 3**:
- Experimenting with ensemble techniques (Bagging, Boosting) to improve prediction accuracy

---

## Introduction to Ensemble Methods

**Ensemble methods** combine multiple models to improve performance:
- **Bagging**: Parallel training, reduces variance (e.g., Random Forest)
- **Boosting**: Sequential training, reduces bias (e.g., AdaBoost, Gradient Boosting)


## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- The real Breast Cancer Wisconsin (Diagnostic) dataset bundled with scikit-learn:
  569 biopsies described by 30 measurements, labelled malignant or benign — no download

**Outputs:** What you'll see when you run the cells

- Held-out accuracy for one decision tree and for three ensembles built from trees,
  plus the gap between each ensemble and the single tree it is made of.

---

In [1]:
# CELL: Import a single tree plus three ways of combining many trees.
# WHY: the whole lesson is one experiment - does a TEAM of weak models
# (bagging, AdaBoost, gradient boosting) beat one tree working alone?
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score

In [2]:
# Load a REAL diagnostic dataset: 569 breast-tumour biopsies (Wisconsin Diagnostic, UCI),
# each described by 30 measurements taken from a digitized image of the cell nuclei.
# WHY this one: the classes genuinely overlap, so a single tree has real errors to fix —
# which is the only way an ensemble can show what it is worth.
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target        # 0 = malignant, 1 = benign

# stratify=y keeps the malignant/benign ratio identical in both halves of the split.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Real dataset: {X.shape[0]} biopsies x {X.shape[1]} measurements")
print(f"Classes: {np.bincount(y)} (malignant, benign)")
print(f"Training set: {X_train.shape}, Test set: {X_test.shape}")
print(f"Always-predict-'benign' baseline on the test set: {(y_test == 1).mean():.4f}")

Real dataset: 569 biopsies x 30 measurements
Classes: [212 357] (malignant, benign)
Training set: (455, 30), Test set: (114, 30)
Always-predict-'benign' baseline on the test set: 0.6316


## Part 1: Base Model (Single Decision Tree)


In [3]:
# Base model: a single decision tree (our reference point)
base_tree = DecisionTreeClassifier(random_state=42)
base_tree.fit(X_train, y_train)
base_pred = base_tree.predict(X_test)
base_acc = accuracy_score(y_test, base_pred)

print(f"Single Decision Tree accuracy: {base_acc:.4f}")
print(f"  ...on the TRAINING data it scores {base_tree.score(X_train, y_train):.4f} —")
print("  an unpruned tree can always memorize its training set. The gap between those")
print("  two numbers is the variance the ensembles below are going to attack.")

Single Decision Tree accuracy: 0.9123
  ...on the TRAINING data it scores 1.0000 —
  an unpruned tree can always memorize its training set. The gap between those
  two numbers is the variance the ensembles below are going to attack.


## Part 2: Bagging (Random Forest is a type of Bagging)


In [4]:
# Bagging Classifier: many trees trained in parallel on bootstrap samples
bagging = BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=100,
                            random_state=42)
bagging.fit(X_train, y_train)
bagging_pred = bagging.predict(X_test)
bagging_acc = accuracy_score(y_test, bagging_pred)

print(f"Bagging (100 trees) accuracy:  {bagging_acc:.4f}")
print(f"Improvement over single tree:  {bagging_acc - base_acc:+.4f}")

Bagging (100 trees) accuracy:  0.9386
Improvement over single tree:  +0.0263


## Part 3: Boosting Methods


In [5]:
# AdaBoost: trees trained sequentially, each focusing on previous errors
adaboost = AdaBoostClassifier(n_estimators=100, random_state=42)
adaboost.fit(X_train, y_train)
adaboost_pred = adaboost.predict(X_test)
adaboost_acc = accuracy_score(y_test, adaboost_pred)

# Gradient Boosting: boosting with gradient-based error correction
gboost = GradientBoostingClassifier(n_estimators=100, random_state=42)
gboost.fit(X_train, y_train)
gboost_pred = gboost.predict(X_test)
gboost_acc = accuracy_score(y_test, gboost_pred)

print("Final comparison (test accuracy):")
print("-" * 44)
print(f"  Single Decision Tree: {base_acc:.4f}")
print(f"  Bagging (100 trees):  {bagging_acc:.4f}")
print(f"  AdaBoost (100):       {adaboost_acc:.4f}")
print(f"  Gradient Boosting:    {gboost_acc:.4f}")
print()
print(f"  (Always predicting 'benign' would score {(y_test == 1).mean():.4f})")
print()
print("All three ensembles beat the single tree on this data. Bagging attacks the")
print("tree's VARIANCE by averaging 100 trees grown on bootstrap resamples; the two")
print("boosting methods attack its BIAS by training each tree on what the previous")
print("ones got wrong. Which family wins depends on the dataset - here boosting edges")
print("ahead, but the margin is a handful of biopsies out of 114, so do not read a")
print("universal law into it.")

Final comparison (test accuracy):
--------------------------------------------
  Single Decision Tree: 0.9123
  Bagging (100 trees):  0.9386
  AdaBoost (100):       0.9561
  Gradient Boosting:    0.9561

  (Always predicting 'benign' would score 0.6316)

All three ensembles beat the single tree on this data. Bagging attacks the
tree's VARIANCE by averaging 100 trees grown on bootstrap resamples; the two
boosting methods attack its BIAS by training each tree on what the previous
ones got wrong. Which family wins depends on the dataset - here boosting edges
ahead, but the margin is a handful of biopsies out of 114, so do not read a
universal law into it.


## Summary

### Key Concepts:
1. **Bagging**: Trains models in parallel, averages predictions (reduces variance)
2. **Boosting**: Trains models sequentially, focuses on errors (reduces bias)
3. **Ensemble Benefit**: on these 569 real biopsies, every ensemble beat the single tree
4. **When to use**: Bagging for high variance models, Boosting for high bias models

### A caution about small test sets
The test set here is 114 biopsies, so one extra correct prediction moves accuracy by
0.0088. Differences between the three ensembles are within a few such steps of each
other — real, but not a ranking you should carry to another dataset. Unit 5 shows how
cross-validation gives a sturdier comparison.

**Reference:** Course 04, Unit 3: "Experimenting with ensemble techniques (Bagging, Boosting)"


## 📚 References

1. Breiman, L. (1996). *Bagging Predictors*. Machine Learning, 24, 123–140.
2. Freund, Y., & Schapire, R. E. (1997). *A Decision-Theoretic Generalization of On-Line Learning and an Application to Boosting*. Journal of Computer and System Sciences, 55(1), 119–139.
3. Chen, T., & Guestrin, C. (2016). *XGBoost: A Scalable Tree Boosting System*. KDD 2016. <https://arxiv.org/abs/1603.02754>
4. Bentéjac, C., Csörgő, A., & Martínez-Muñoz, G. (2021). *A Comparative Analysis of XGBoost* (journal version: Artificial Intelligence Review, 54, 1937–1967). <https://arxiv.org/abs/1911.01914>
